In [ ]:
# 1. 필요한 라이브러리 설치
!pip -q install easyocr openpyxl pandas pillow opencv-python-headless anthropic


In [ ]:
# 2. 라이브러리 불러오기
import os
import re
import cv2
import easyocr
import numpy as np
import pandas as pd
from PIL import Image
from google.colab import files
from IPython.display import display


In [ ]:
# 3. 어려운 단어 데이터셋 업로드 및 확인
uploaded = files.upload()
excel_files = [name for name in uploaded if name.lower().endswith((".xlsx", ".xls"))]
if not excel_files:
    raise ValueError("엑셀 데이터셋(.xlsx 또는 .xls)을 업로드해 주세요.")

excel_file = excel_files[0]
df = pd.read_excel(excel_file)
print("데이터 개수:", len(df))
print("컬럼 이름:", df.columns.tolist())
display(df.head())


In [ ]:
# 4. OCR 모델 준비 (한국어 + 영어)
reader = easyocr.Reader(["ko", "en"], gpu=False)
print("OCR 준비 완료!")


In [ ]:
# 5. 이미지 업로드 후 글자 인식 (OCR)
print("글자가 포함된 이미지 파일(PNG, JPG, JPEG)을 업로드해 주세요.")
uploaded_images = files.upload()
image_files = [name for name in uploaded_images
               if os.path.splitext(name)[1].lower() in [".png", ".jpg", ".jpeg"]]
if not image_files:
    raise ValueError("이미지 파일(.png, .jpg, .jpeg)을 업로드해 주세요.")

image_file = image_files[0]
image = Image.open(image_file).convert("RGB")
display(image)

img = np.array(image)
gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
gray = cv2.resize(gray, None, fx=1.5, fy=1.5, interpolation=cv2.INTER_CUBIC)
results = reader.readtext(gray, detail=1)

print("\n--- 개별 인식 결과 ---")
for box, text, confidence in results:
    print(f"{text} (정확도: {confidence:.2f})")

ocr_text = " ".join(text for box, text, confidence in results)
full_text = ocr_text
print("\n--- 전체 인식 결과 ---")
print(ocr_text)


In [ ]:
# 6. OCR 결과에서 데이터셋의 어려운 단어 찾기
TERM_COL = "어려운 단어"
EXPLAIN_COL = "쉬운 단어"

if TERM_COL not in df.columns or EXPLAIN_COL not in df.columns:
    raise ValueError(
        f"엑셀에 '{TERM_COL}', '{EXPLAIN_COL}' 열이 있는지 확인해 주세요. "
        f"현재 열: {df.columns.tolist()}"
    )

def normalize(text):
    return re.sub(r"\s+", "", str(text)).strip()

dictionary = {}
for _, row in df.iterrows():
    term = str(row[TERM_COL]).strip()
    explanation = str(row[EXPLAIN_COL]).strip()
    if term and term.lower() != "nan" and explanation and explanation.lower() != "nan":
        dictionary[normalize(term)] = {"용어": term, "쉬운 설명": explanation}

normalized_ocr = normalize(ocr_text)
unique_matches = {}
for key in sorted(dictionary.keys(), key=len, reverse=True):
    if len(key) >= 2 and key in normalized_ocr:
        item = dictionary[key]
        unique_matches[item["용어"]] = item["쉬운 설명"]

if unique_matches:
    print("--- 데이터셋 매칭 결과 ---")
    for term, explanation in unique_matches.items():
        print(f"어려운 용어: {term}")
        print(f"쉬운 설명: {explanation}")
        print("-" * 30)
else:
    print("데이터셋에서 일치하는 어려운 용어를 찾지 못했습니다.")


In [ ]:
# 7. (선택) 웹캠으로 사진 촬영 후 OCR 실행
# 웹캠을 사용하지 않는다면 이 셀은 건너뛰어도 됩니다.
from IPython.display import Javascript, display
from google.colab.output import eval_js
from base64 import b64decode

def take_photo(filename="webcam_photo.jpg", quality=0.8):
    js = Javascript("""
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capture';
      div.appendChild(capture);
      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});
      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
      await new Promise((resolve) => capture.onclick = resolve);
      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    """)
    display(js)
    data = eval_js(f"takePhoto({quality})")
    binary = b64decode(data.split(",")[1])
    with open(filename, "wb") as f:
        f.write(binary)
    return filename

try:
    print("카메라가 켜지면 화면을 보고 Capture 버튼을 눌러주세요.")
    image_path = take_photo()
    webcam_results = reader.readtext(image_path)
    full_text = " ".join(item[1] for item in webcam_results)
    ocr_text = full_text
    print("\n--- 촬영한 사진에서 추출한 텍스트 ---")
    print(full_text)
except Exception as err:
    print("카메라 실행에 실패했습니다. 카메라 권한과 웹캠 연결을 확인해 주세요.")
    print("오류 내용:", err)


In [ ]:
# 8. (선택) Claude API로 어려운 표현을 쉬운 말로 바꾸기
# Colab 왼쪽의 비밀 정보(열쇠 아이콘)에 ANTHROPIC_API_KEY를 등록하세요.
# API 키를 코드에 직접 입력하거나 GitHub에 올리지 마세요.
try:
    from google.colab import userdata
    import anthropic

    api_key = userdata.get("ANTHROPIC_API_KEY")
    if not api_key:
        raise ValueError("Colab 비밀 정보에 ANTHROPIC_API_KEY가 등록되어 있지 않습니다.")

    client = anthropic.Anthropic(api_key=api_key)
    prompt = (
        "아래 글에서 70대 어르신이 이해하기 어려운 단어를 찾아 "
        "각각 쉬운 말로 바꿔 주세요.\n\n글: " + full_text
    )
    msg = client.messages.create(
        model="claude-3-5-sonnet-20241022",
        max_tokens=1000,
        messages=[{"role": "user", "content": prompt}],
    )
    print(msg.content[0].text)
except Exception as e:
    print("Claude API 실행에 실패했습니다. API 키 등록 여부와 모델 사용 가능 여부를 확인해 주세요.")
    print("오류 내용:", e)
